# ProSeg on the Xenium breast cancer dataset (xenium_2)

This notebook runs **ProSeg** on the Xenium human breast cancer dataset.

- **Input**: `transcripts.parquet` from the Xenium output bundle
- **Prior**: the `cell_id` column of `transcripts.parquet`, i.e. the official 10x Genomics segmentation
- **Output**: cell metadata, transcript metadata, and cell polygons written to the output directory

The notebook skips the run if a finished result already exists, so it can be re-executed safely.

## 1. Configuration

Edit the paths below to match your environment.

In [ ]:
import os
import subprocess

import pandas as pd

BASE     = "yourpath/proseg"
OUT_DIR  = os.path.join(BASE, "output", "xenium_2")
TX_PATH  = "yourpath/dataset/xenium_breast/transcripts.parquet"
LOG_PATH = os.path.join(OUT_DIR, "proseg.log")

N_THREADS = 12
EXPECTED_CELLS_IN_ROI = 29_892   # reference cells in the ground-truth ROI, for comparison only

os.makedirs(OUT_DIR, exist_ok=True)
CELL_METADATA = os.path.join(OUT_DIR, "cell-metadata.csv.gz")

## 2. Check the input

In [ ]:
if not os.path.exists(TX_PATH):
    raise FileNotFoundError(f"Transcript file not found: {TX_PATH}")
print(f"Found transcript file: {TX_PATH}")

## 3. Check for an existing result

If `cell-metadata.csv.gz` already exists, the run is skipped. Delete the file to run ProSeg again.

In [ ]:
already_done = os.path.exists(CELL_METADATA)
if already_done:
    n_cells = len(pd.read_csv(CELL_METADATA))
    print(f"[Skip] cell-metadata.csv.gz already exists ({n_cells:,} cells). Delete it to re-run.")
else:
    print("No existing result found. ProSeg will be run in the next step.")

## 4. Run ProSeg

The full command is printed before execution, and all ProSeg output is written to the log file.

In [ ]:
cmd = [
    "proseg", "--xenium",
    "--nthreads", str(N_THREADS),
    "--output-cell-metadata",       CELL_METADATA,
    "--output-transcript-metadata", os.path.join(OUT_DIR, "transcript-metadata.csv.gz"),
    "--output-cell-polygons",       os.path.join(OUT_DIR, "cell-polygons.geojson.gz"),
    "--overwrite",
    TX_PATH,
]

print("ProSeg on Xenium breast cancer")
print(f"  Input : {TX_PATH}")
print(f"  Output: {OUT_DIR}")
print(f"  Log   : {LOG_PATH}")
print("\n  " + " \\\n    ".join(cmd))

if already_done:
    print("\n[Skip] An existing result was found, so ProSeg was not run.")
    returncode = 0
else:
    with open(LOG_PATH, "w") as log:
        result = subprocess.run(cmd, cwd=OUT_DIR, stdout=log, stderr=log)
    returncode = result.returncode

## 5. Summarize the result

In [ ]:
if returncode == 0:
    n_cells = len(pd.read_csv(CELL_METADATA))
    print(f"Done. Number of cells: {n_cells:,} (about {EXPECTED_CELLS_IN_ROI:,} reference cells in the ground-truth ROI)")
    print(f"  -> {CELL_METADATA}")
else:
    raise RuntimeError(f"ProSeg exited with code {returncode}. See the log: {LOG_PATH}")